In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.insert(0, "../")

from data.features import (
    agg_crops,
    agg_surplus,
    agg_weather,
    agg_weather_w_lag,
    daily_nitrate,
    lagged_sensor_nitrate,
    nitrate_rolling,
    nitrate_avg_seasonal,
    nitrate_avg_calendar,
    doy_climatology_pure_signal,
)
from data.transforms import flatten_buckets, merge_on_date, match_seasonal
from data import get_site_ids

## Question: Is lagged weather data helpful?

In [ ]:
from cook import *
from recipes2 import _covariates


def recipe_lagger(lags=[]):
    def recipe(site_uid):
        def lagged(lag):
            wdf = (agg_weather(site_uid, edges=[])
                   .sort_values("date").set_index("date").asfreq("D")  # regular daily index
                   .shift(lag))                                        # actually lag by `lag` days
            wdf.columns = [f"{c}_lag{lag}" for c in wdf.columns]       # suffix the value columns
            return wdf.reset_index()
        n_daily, parts = _covariates(site_uid)
        parts += [lagged(i) for i in lags]
        out = merge_on_date([n_daily, *parts], spine=n_daily.index)
        return out.dropna(subset=["nitrate_con"]).reset_index(drop=True)
    return recipe

recipe = recipe_lagger([1])

lags = [1, 2, 3, 7, 10, 14, 21, 30]
recipes = {f"Lags {lags[:i]}" : recipe_lagger(lags[:i]) for i in range(len(lags))}
print(compare_many(recipes, **FAST_XGB))
    

In [1]:
from cook import *
from recipes2 import _covariates
from data.features import agg_weather, nitrate_violations   # names the recipe needs
from data.transforms import merge_on_date                   # not re-exported by `from cook import *`

def recipe_lagger(lags=[]):
    def recipe(site_uid):
        def lagged(lag):
            wdf = (agg_weather(site_uid, edges=[])
                   .sort_values("date").set_index("date").asfreq("D")
                   .shift(lag))
            wdf.columns = [f"{c}_lag{lag}" for c in wdf.columns]
            return wdf.reset_index()
        n_daily, parts = _covariates(site_uid)
        parts += [lagged(i) for i in lags]
        v = nitrate_violations(site_uid, threshold=10).rename("violation")
        out = merge_on_date([v, *parts], spine=n_daily.index)
        return out.dropna(subset=["violation"]).reset_index(drop=True)
    return recipe

lags = [1, 2, 3, 7, 10, 14, 21, 30]
recipes = {f"Lags {lags[:i]}": recipe_lagger(lags[:i]) for i in range(len(lags))}
print(compare_many(recipes, target="violation", task="clf", **FAST_XGB))   # both args set


compare_many: done 8/8 recipes in 6594s                              
                               n_sites  n_rows  n_feat  loso_auc  lobo_auc  \
recipe                                                                       
Lags []                             80  160074      21  0.806219  0.774217   
Lags [1]                            80  160074      30  0.810644  0.771240   
Lags [1, 2]                         80  160074      39  0.811065  0.776148   
Lags [1, 2, 3]                      80  160074      48  0.811180  0.773234   
Lags [1, 2, 3, 7]                   80  160074      57  0.814095  0.779225   
Lags [1, 2, 3, 7, 10]               80  160074      66  0.815917  0.776278   
Lags [1, 2, 3, 7, 10, 14]           80  160074      75  0.815311  0.782871   
Lags [1, 2, 3, 7, 10, 14, 21]       80  160074      84  0.818340  0.779600   

                                  prauc     brier      base  between_rate_r2  \
recipe                                                               

In [1]:
import pandas as pd
pd.read_csv("test3.csv")

,recipe,n_sites,n_rows,n_feat,loso_r2,lofo_r2,rmse,between_r2,within_r2,macro_r2
0,test,3,5775,30,0.278198,0.298703,3.522577,0.585294,0.259444,0.214513


In [1]:
import sys

sys.path.insert(0, "../")

from data.features import (
    agg_crops,
    agg_surplus,
    agg_weather,
    agg_weather_w_lag,
    daily_nitrate,
    lagged_sensor_nitrate,
    nitrate_rolling,
    nitrate_avg_seasonal,
    nitrate_avg_calendar,
    doy_climatology_pure_signal,
)
from data.transforms import flatten_buckets, merge_on_date, match_seasonal
from data import get_site_ids

from cook import *
from recipes2 import _covariates


def recipe_lagger(lags=[]):
    def recipe(site_uid):
        def lagged(lag):
            wdf = (
                agg_weather(site_uid, edges=[])
                .sort_values("date")
                .set_index("date")
                .asfreq("D")  # regular daily index
                .shift(lag)
            )  # actually lag by `lag` days
            wdf.columns = [f"{c}_lag{lag}" for c in wdf.columns]  # suffix the value columns
            return wdf.reset_index()

        n_daily, parts = _covariates(site_uid)
        parts += [lagged(i) for i in lags]
        out = merge_on_date([n_daily, *parts], spine=n_daily.index)
        return out.dropna(subset=["nitrate_con"]).reset_index(drop=True)

    return recipe


recipes = {"test": recipe_lagger([1])}
sites = ["WQS0115", "WQS0039", "WQS0003"]

# lags = [1, 2, 3, 7, 10, 14, 21, 30]
# recipes = {f"Lags {lags[:i]}": recipe_lagger(lags[:i]) for i in range(len(lags))}
test = compare_many(
    recipes,
    sites=sites,
    **FAST_XGB,
)

compare_many: done 1/1 recipes in 19s                              


In [4]:
importance = pd.read_csv("test_results/experiment_weather_lag_importance.csv")
print(importance)

                  Unnamed: 0      test
0                   Soybeans  0.076590
1                    Alfalfa  0.075393
2                    doy_sin  0.068178
3                    doy_cos  0.065615
4                      Other  0.064542
5               surplus_kgha  0.057802
6   fuel_moisture_1000h_lag1  0.056667
7                Hay_Pasture  0.056359
8                 total_kg_N  0.053151
9               Small_Grains  0.052434
10                      Corn  0.050122
11       fuel_moisture_1000h  0.037706
12                    Fallow  0.035597
13                     Nonag  0.035241
14                 solar_rad  0.029864
15             min_temp_lag1  0.027257
16            solar_rad_lag1  0.025878
17                  min_temp  0.022892
18             max_temp_lag1  0.013063
19                  max_temp  0.010211
20                       vpd  0.009776
21     min_rel_humidity_lag1  0.009648
22        evapotranspiration  0.009299
23                  vpd_lag1  0.008669
24   evapotranspiration_l